# Local Friendliness (semi-Brukner) Inequality Violation

This is a follow-up to the "Wigner's friend" interference-erasure experiment: instead of a simple test for whether interference is preserved, this notebook implements a full **Local Friendliness (LF) test** (Bong, Utreras-Alarcón, Ghafari, Liang, Tischler, Cavalcanti, Pryde, Wiseman, *Nature Physics* 16, 1199 (2020); gate-based realization following Zeng, Labib, Russo, arXiv:2409.15302).

**Idea.** Local Friendliness (LF) is the conjunction of two intuitively reasonable assumptions:
- **AOE** (Absoluteness of Observed Events) — every observed event happens the same way for all observers;
- **LA** (Local Agency) — a free choice of measurement setting does not correlate with events outside its light cone.

If both assumptions hold, a particular combination of correlations (the **semi-Brukner inequality**) cannot exceed 2. Quantum mechanics predicts this combination can reach up to **2√2 ≈ 2.828** (the Tsirelson bound — the same limit that appears in standard Bell tests).

**Scheme (simplified single-friend variant, Section 2.3 of arXiv:2409.15302):**
- Alice and her friend Charlie share a singlet state with Bob;
- Charlie unitarily "measures" his half (a CNOT onto an ancilla qubit);
- Alice chooses between **PEEK** (x=1, simply read Charlie's recorded outcome) or **REVERSE** (x=3, undo Charlie's measurement and measure the system in a rotated basis θ₃);
- Bob always measures his half directly, in one of two rotated bases (β₂ or β₃), with no friend of his own.

**Important honesty note.** This is a loophole-prone proof-of-concept — there is no space-like separation between Alice/Charlie and Bob, and the shot count is modest. It is not a rigorous, loophole-free test, but an illustrative demonstration of the same kind as arXiv:2409.15302. The inequality violation says nothing about which interpretation of quantum mechanics (Many-Worlds, Copenhagen, etc.) is correct — it shows only that **AOE and LA together are incompatible with quantum mechanics**; which of the two to abandon remains an open interpretational question.

## Step 1. Install Qiskit

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime -q

## Step 2. Circuit and local-simulator sanity check

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile

theta1 = 0.0
theta3 = np.pi/2
beta2  = np.pi/4
beta3  = 3*np.pi/4


def prepare_singlet(qc, qc_idx, qd_idx):
    """(|01> - |10>) / sqrt(2)"""
    qc.h(qc_idx)
    qc.cx(qc_idx, qd_idx)
    qc.z(qc_idx)
    qc.x(qd_idx)


def rotate_measure(qc, qubit, cbit, angle):
    """Measurement in the RY(angle)|0>, RY(angle)|1> basis -- consistent with PEEK's Z basis at angle=0."""
    qc.ry(-angle, qubit)
    qc.measure(qubit, cbit)


def build_circuit(x_setting, y_angle):
    """q0 = S_C (Alice's half), q1 = S_D (Bob's half), q2 = Charlie (Alice's friend)"""
    qc = QuantumCircuit(3, 2)
    prepare_singlet(qc, 0, 1)
    qc.cx(0, 2)

    if x_setting == 'PEEK':
        qc.measure(2, 0)
    elif x_setting == 'REVERSE3':
        qc.cx(0, 2)
        rotate_measure(qc, 0, 0, theta3)
    else:
        raise ValueError

    rotate_measure(qc, 1, 1, y_angle)
    return qc


def counts_to_expectation(counts, shots):
    E = 0
    for bitstr, n in counts.items():
        b = 1 - 2*int(bitstr[0])
        a = 1 - 2*int(bitstr[1])
        E += a*b*n
    return E/shots


settings = [('PEEK', beta2), ('PEEK', beta3), ('REVERSE3', beta2), ('REVERSE3', beta3)]
labels = ['<A1B2>', '<A1B3>', '<A3B2>', '<A3B3>']
circuits = [build_circuit(x, y) for x, y in settings]

from qiskit_aer import AerSimulator
backend_sim = AerSimulator()
shots = 20000
transpiled = transpile(circuits, backend_sim)
results = backend_sim.run(transpiled, shots=shots).result()

E = [counts_to_expectation(results.get_counts(i), shots) for i in range(4)]
for lbl, val in zip(labels, E):
    print(f"{lbl} = {val:.4f}")

S_sim = -E[0] + E[1] - E[2] - E[3]
print(f"\nSemi-Brukner S (simulator) = {S_sim:.4f}  (classical bound: 2, quantum max: 2.8284)")

## Step 3. Connect to IBM Quantum

1. Go to https://quantum.ibm.com and sign up / log in.
2. Copy your API token.
3. Paste it below and run the cell (once).

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    overwrite=True,
)
print("Account saved.")

## Step 4. Run on real hardware

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Selected processor: {backend.name}")

transpiled_hw = transpile(circuits, backend=backend, optimization_level=1)
sampler = Sampler(backend)
job = sampler.run(transpiled_hw, shots=8000)
print(f"Job submitted, id: {job.job_id()}")
res = job.result()

E_hw = []
for i in range(4):
    counts = res[i].data.c.get_counts()
    total = sum(counts.values())
    Ei = 0
    for bitstr, n in counts.items():
        b = 1 - 2*int(bitstr[0])
        a = 1 - 2*int(bitstr[1])
        Ei += a*b*n
    E_hw.append(Ei/total)

for lbl, val in zip(labels, E_hw):
    print(f"{lbl} = {val:.4f}")

S_hw = -E_hw[0] + E_hw[1] - E_hw[2] - E_hw[3]
print(f"\nSemi-Brukner S (real hardware, {backend.name}) = {S_hw:.4f}")
print("Classical LF bound: S <= 2. Quantum theoretical max: 2.8284.")
if S_hw > 2:
    print(f"\n>>> Local Friendliness inequality violation confirmed on real hardware! <<<")
else:
    print("\nNo violation observed in this run -- possibly due to hardware noise.")
print("\nReminder: this is not a loophole-free test and does not select between")
print("interpretations of QM -- it only shows that AOE and LA together are")
print("incompatible with quantum mechanics.")